# GlyphGAN

A DCGAN trained on rasterised glyphs, with a latent-space interpolation
video as the output.

The model lives in `glyphgan.py`; this notebook just drives it.


## Dataset

Any folder of images laid out for `ImageFolder` works. To build one from the
fonts installed on a Mac, use [fontscrape](https://github.com/latentcollection/macOS-fontface-scraper):

```sh
fontscrape --glyphs a --size 64 --mode xheight --per-family 3 --manifest --out dataset/
```

`--glyphs` (plural) writes one subdirectory per character, which is the layout
`ImageFolder` expects. `--per-family 3` stops whichever families you own the
most weights of from dominating the latent space.


In [ ]:
import glyphgan as gg

img_dir = "dataset/"
size = 64

gg.pick_device()

## Train

The budget is in steps, not epochs. A few hundred glyphs make an epoch only a
handful of steps, so an epoch count borrowed from a larger dataset trains for
almost no time at all.

Checkpoints land in `checkpoints/` every few thousand steps and on exit,
including Ctrl-C. Re-running this cell resumes from the latest one.


In [ ]:
gen, dis = gg.train(
    img_dir,
    size=size,
    max_steps=20000,
    batch_size=32,
    save_every_steps=2000,
)

## Render the interpolation

This reads a checkpoint from disk and needs nothing from the training cell,
so a video can be rendered at any point, from any run.


In [ ]:
gg.render_interpolation(
    "checkpoints/generator.pt",
    "glyphgan-render.mp4",
    keys=8,
    frames=60,
    fps=30,
    seed=0,
)

## Preview a few samples


In [ ]:
import matplotlib.pyplot as plt
import torch

g = gg.load_generator("checkpoints/generator.pt")

with torch.no_grad():
    z = torch.randn(8, gg.LATENT, device=gg.pick_device())
    imgs = [gg.to_image(x) for x in g(z)]

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for ax, im in zip(axes, imgs):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255)
    ax.axis("off")
plt.show()